In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder, FunctionTransformer
from sklearn.model_selection import StratifiedKFold, train_test_split
from category_encoders import CountEncoder
from funcs import datetime_processing_func, to_categorical_func, fill_unknown
from sklearn.model_selection import cross_validate
from sklearn.base import clone

In [10]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna

In [11]:
df = pd.read_csv("./fraudTrain.csv")
df_test = pd.read_csv("./fraudTest.csv")

In [12]:
results_imp = joblib.load("results_imp.joblib")
pipe_for_rndf = joblib.load("results_baseline.joblib")["pipeline"]

In [13]:
pipe_imp = results_imp["pipeline"]
metrics_imp = results_imp["metrics"]

In [14]:
pipe_for_rndf.steps.insert(1, ("to_categorical", pipe_imp[1]))
pipe_for_rndf

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('datetime_proc', ...), ('to_categorical', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function dat...0022BF36CB060>
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword arguments to pass to func... versionadded:: 0.18",{'date_feature': 'trans_date_trans_time'}
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` met

In [15]:
metrics_imp

{'fit_time': np.float64(6.4785576343536375),
 'score_time': np.float64(1.1306095123291016),
 'test_accuracy': np.float64(0.9975635168910367),
 'test_precision': np.float64(0.9890169371309984),
 'test_recall': np.float64(0.37296037296037293),
 'test_f1': np.float64(0.5414900175440452),
 'test_roc_auc': np.float64(0.9934501466342924)}

In [16]:
df = df.drop_duplicates()

personal_info = ["first", "last", "dob", "cc_num", "street", "city", "zip", "dob", "trans_num", "is_fraud", "Unnamed: 0"]

y_train, y_test = df["is_fraud"], df_test["is_fraud"]
X_train, X_test = df.drop(personal_info, axis =1), df_test.drop(personal_info, axis =1)

cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
folds = []
folds_for_rndf = []

cat_cols = ['merchant', 'category', 'gender', 'state', 'job']
train_categories = {c: sorted(X_train[c].dropna().unique()) for c in cat_cols}
pipe_imp.steps.insert(2, ("fillna", FunctionTransformer(fill_unknown, kw_args={"cols" : cat_cols})))

pipe_imp.set_params(to_categorical__kw_args={"cols": cat_cols, "categories": train_categories})
pipe_imp.set_params(to_categorical__kw_args={"cols": cat_cols, "categories": train_categories})

for step, (tr, vl) in enumerate(cv.split(X_train, y_train)):
    pipe = clone(pipe_imp)
    X_tr = pipe.fit_transform(X_train.iloc[tr], y_train.iloc[tr])
    X_vl = pipe.transform(X_train.iloc[vl])
    folds.append([X_tr, y_train.iloc[tr].to_numpy(), X_vl, y_train.iloc[vl].to_numpy()])

for step, (tr, vl) in enumerate(cv.split(X_train, y_train)):
    pipe1 = clone(pipe_for_rndf)
    X_tr = pipe1.fit_transform(X_train.iloc[tr], y_train.iloc[tr])
    X_vl = pipe1.transform(X_train.iloc[vl])
    folds_for_rndf.append([X_tr, y_train.iloc[tr].to_numpy(), X_vl, y_train.iloc[vl].to_numpy()])

In [17]:
def undersample(X, y, neg_frac=0.1, seed=42):
    rng = np.random.default_rng(seed)
    is_pos = y == 1
    keep = is_pos | (rng.random(len(y)) < neg_frac)
    return X[keep], y[keep]

folds_fast = []
for X_tr, y_tr, X_vl, y_vl in folds_for_rndf:
    X_s, y_s = undersample(X_tr, y_tr)
    folds_fast.append((X_s, y_s, X_vl, y_vl))

In [18]:
folds[0][0]

,merchant,category,amt,gender,state,lat,long,city_pop,job,unix_time,merch_lat,merch_long,Date,Time
0,"fraud_Rippin, Kub and Mann",misc_net,4.97,F,NC,36.0788,-81.1781,3495,"Psychologist, counselling",1325376018,36.011293,-82.048315,1,0
1,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,F,WA,48.8878,-118.2105,149,Special educational needs teacher,1325376044,49.159047,-118.186462,1,0
2,fraud_Lind-Buckridge,entertainment,220.11,M,ID,42.1808,-112.2620,4154,Nature conservation officer,1325376051,43.150704,-112.154481,1,0
3,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,M,MT,46.2306,-112.1138,1939,Patent attorney,1325376076,47.034331,-112.561071,1,1
5,"fraud_Stroman, Hudson and Erdman",gas_transport,94.63,F,PA,40.3750,-75.2045,2158,Transport planner,1325376248,40.653382,-76.152667,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296666,"fraud_Hyatt, Russel and Gleichner",health_fitness,7.30,F,IL,41.0646,-87.5917,2135,"Psychotherapist, child",1371816562,40.556811,-88.092339,173,729
1296668,"fraud_Metz, Russel and Metz",kids_pets,100.85,F,MS,32.1530,-90.1217,19685,Fine artist,1371816683,31.377697,-90.528450,173,731
1296669,fraud_Stiedemann Inc,misc_pos,37.38,F,NE,41.4972,-98.7858,509,"Nurse, children's",1371816696,41.728638,-99.039660,173,731
1296671,fraud_Abernathy and Sons,food_dining,51.70,M,MD,39.2667,-77.5101,100,"Production assistant, television",1371816739,38.906881,-78.246528,173,732


In [19]:
folds_for_rndf

[[         gender_M  merchant  category       job     state     amt      lat  \
  0             0.0  0.015274  0.048812  0.002200  0.005439    4.97  36.0788   
  1             0.0  0.009436  0.095439  0.002477  0.004990  107.23  48.8878   
  2             1.0  0.001626  0.072563  0.018508  0.002858  220.11  42.1808   
  3             1.0  0.003562  0.101573  0.009872  0.002151   45.00  46.2306   
  5             0.0  0.002337  0.101573  0.011878  0.005929   94.63  40.3750   
  ...           ...       ...       ...       ...       ...     ...      ...   
  1296666       0.0  0.000906  0.066068  0.006277  0.005361    7.30  41.0646   
  1296668       0.0  0.001392  0.087144  0.006173  0.005597  100.85  32.1530   
  1296669       0.0  0.004986  0.061456  0.018661  0.007563   37.38  41.4972   
  1296671       1.0  0.002654  0.070426  0.019480  0.006114   51.70  39.2667   
  1296673       1.0  0.000847  0.070426  0.001531  0.006060   74.90  43.3526   
  
               long  city_pop   unix_

In [20]:
cat_features = ['merchant', 'category', 'gender', 'state', 'job']

In [21]:
models = {
    "rndforest" : RandomForestClassifier(n_jobs=-1, class_weight='balanced_subsample'),
    "gradboost" : GradientBoostingClassifier(),
    "lgbm" : LGBMClassifier(n_estimators=400, subsample_freq=1, min_child_weight=1e-3, min_split_gain=0.0, cat_smooth=20, min_data_per_group=200, n_jobs=-1, random_state=42, verbosity=-1),
    "xgboost" : XGBClassifier(device="cuda",n_jobs=-1, enable_categorical=True, random_state = 42),
    "catboost" : CatBoostClassifier(task_type="GPU", devices="0", thread_count=-1, verbose = 0, iterations=2000, border_count=64)
}

In [22]:
def rndforest(trial):
    return {"max_depth": trial.suggest_int("max_depth", 6, 30),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50, log=True),
            "max_features": trial.suggest_float("max_features", 0.1, 0.8),
            "class_weight": trial.suggest_categorical("class_weight", [None, "balanced_subsample"]),
           }
def lgbm(trial):
    return {"learning_rate" : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "n_estimators" : trial.suggest_int("n_estimators", 100, 1000, step=100),
            "min_child_samples" : trial.suggest_int("min_child_samples", 100, 1000, step=100),
            "num_leaves": trial.suggest_int("num_leaves", 15, 255, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
           }
def xgboost(trial):
    return {"learning_rate" : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "n_estimators" : trial.suggest_int("n_estimators", 100, 1000, step=100),
            "max_depth" : trial.suggest_int("max_depth", 3, 10), 
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
           }
def catboost(trial):
    return {"learning_rate" : trial.suggest_float("learning_rate", 0.01, 0.3, log=True), 
            "depth": trial.suggest_int("depth", 6, 10),
            "l2_leaf_reg": trial.suggest_int("l2_leaf_reg", 1, 30, log=True),
            "random_strength": trial.suggest_int("random_strength", 0, 5),
           }
param_grids = {
    "rndforest" : rndforest,
    "lgbm": lgbm,
    "xgboost": xgboost,
    "catboost": catboost,
}

In [23]:
from sklearn.metrics import f1_score, accuracy_score, precision_recall_curve, precision_score, recall_score, roc_auc_score, get_scorer_names

In [ ]:
def objective(trial, model_name, model):
    params = param_grids[model_name](trial)
    model = clone(model).set_params(**params)
    scores = []
    curr_folds = folds_fast if (model_name == "rndforest") else folds
    fit_kwargs = {"cat_features": cat_features} if model_name == "catboost" else {}
    for step, (X_tr, y_tr, X_vl, y_vl) in enumerate(curr_folds):
        model.fit(X_tr, y_tr, **fit_kwargs)
        scores.append(f1_score(model.predict(X_vl), y_vl))
        trial.report(np.mean(scores), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(scores)

studies = {}

for model_name, model in param_grids.items():
    model = models[model_name]

    study = optuna.create_study(direction = "maximize", study_name=model_name,
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )
    study.optimize(
        lambda trial: objective(trial, model_name, model),
        n_trials=30,
        timeout=1800,
    )
    studies[model_name] = study
    print(f"{model_name}: F1 = {study.best_value:.4f}, {study.best_params}")

In [ ]:
hptuned_models = {}
hptuned_models = {key : models[key].set_params(**val.best_params) for key, val in studies.items()}
hptuned_models

In [ ]:
results_hptun = {
    "pipeline" : pipe_imp,
    "pipeline_rndf" : pipe_for_rndf,
    "models" : hptuned_models
}

In [ ]:
joblib.dump(results_hptun, "results_hptun.joblib")

In [24]:
results_hptun_loaded = joblib.load("results_hptun.joblib")

In [25]:
hptuned_models = results_hptun_loaded["models"]

In [26]:
hptuned_models

{'rndforest': RandomForestClassifier(max_depth=29, max_features=0.16373376877863927,
                        n_jobs=-1),
 'lgbm': LGBMClassifier(cat_smooth=20, colsample_bytree=0.5585166684709804,
                learning_rate=0.15171221225793705, min_child_samples=700,
                min_data_per_group=200, n_estimators=1000, n_jobs=-1,
                num_leaves=135, random_state=42,
                reg_lambda=0.0014535032128090962, subsample=0.6872909007868266,
                subsample_freq=1, verbosity=-1),
 'xgboost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device='cuda', early_stopping_rounds=None,
               enable_categorical=True, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=0.05681190967117297, max

In [27]:
hptuned_models_check = hptuned_models.copy()
hptuned_models_check = {"catboost" : hptuned_models_check["catboost"]}

In [28]:
pipe_for_rndf

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('datetime_proc', ...), ('to_categorical', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function dat...0022BF36CB060>
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword arguments to pass to func... versionadded:: 0.18",{'date_feature': 'trans_date_trans_time'}
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` met

In [30]:
from sklearn.model_selection import TunedThresholdClassifierCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tuned = {}
hptuned_metrics = {}

for key, model in hptuned_models.items():
    prep = pipe_for_rndf if key == "rndforest" else pipe_imp
    if key == "catboost":
        model = clone(model).set_params(cat_features=tuple(cat_features))
    full_pipe = Pipeline(clone(prep).steps + [("model", clone(model))])
    tuned[key] = TunedThresholdClassifierCV(full_pipe, scoring="f1", cv=cv)
    tuned[key].fit(X_train, y_train)

    proba = tuned[key].predict_proba(X_test)[:, 1]
    predicted = tuned[key].predict(X_test)
    hptuned_metrics[key] = {
        "threshold": tuned[key].best_threshold_,
        "f1_cv": tuned[key].best_score_,
        "accuracy": accuracy_score(y_test, predicted),
        "precision": precision_score(y_test, predicted),
        "recall": recall_score(y_test, predicted),
        "f1": f1_score(y_test, predicted),
        "roc_auc": roc_auc_score(y_test, proba),
    }

hptuned_metrics = pd.DataFrame(hptuned_metrics).T

KeyboardInterrupt: 

In [ ]:
sns.heatmap(hptuned_metrics, annot=True, cmap = "coolwarm")

In [ ]:
hptuned_metrics

После оптимизации threshold наилучшие результаты показали модели catboost и lightgbm, метрика recall, достаточно низкая, но думаю, что для антифрод модели, результат неплохой, учитывая довольно высокий скор precision, обе эти модели дампну в итоговый файл

In [ ]:
tuned["lgbm"].estimator_.steps

In [36]:
joblib.dump(result, "final_result.joblib", compress = 4)

['final_result.joblib']